# CoQLLM — Multi-seed on MovieLens-1M (reviewer-requested variance check)

**Mục đích.** Reviewer #2 (CSA 2026) yêu cầu xác nhận khoảng cách nhỏ so với BinLLM
(+0,0050 AUC / +0,0012 uAUC) không phải nhiễu, bằng cách chạy **nhiều seed ngẫu nhiên**.

**Giao thức (quan trọng — đọc trước khi chạy).**
Ta **tái dùng** MF (R0) + Giai đoạn 1 (R1) + Giai đoạn 2 (R2) — vốn pin seed `2025`, **không đụng tới** —
và **chỉ chạy lại Giai đoạn 3** (Bước 1 LoRA → Bước 2 Q-Former/điều kiện hóa → eval) với 3 seed khác nhau.
Đây là chuẩn "phương sai trên seed tinh chỉnh, bộ mã hóa cố định": vừa rẻ, vừa cô lập đúng
dao động của phần đóng góp (điều kiện hóa, học ở Bước 2).

- Seed dùng: **42, 123, 7** (42 là mặc định/chuẩn; 2 seed thêm là 123 và 7).
- Mỗi seed ghi checkpoint ra thư mục **riêng** nhờ hậu tố `_seed<seed>` (đã sửa trong `configs/config.yaml`),
  nên 3 seed không đè nhau và eval tự bám đúng thư mục.
- Mô hình đánh giá ở đây là **CoQLLM đầy đủ** (arm mặc định `B3-qformer-ucq`) — đúng mô hình so với BinLLM.

**Điều kiện tiên quyết (phải có sẵn, notebook này KHÔNG tạo lại):**
1. Repo tại `/content/CoQLLM`, đang ở branch `exp/multi-seed-movielens`.
2. Môi trường conda `sigllm` + `transformers==4.38.2`, `accelerate==0.27.2` (như notebook reproduce).
3. Dữ liệu đã tiền xử lý + `build_qformer_dataset` đã chạy.
4. Checkpoint **MF / Giai đoạn 1 / Giai đoạn 2** đã có (chạy một lần từ notebook reproduce,
   hoặc khôi phục từ Drive: `ckpt/mf`, `ckpt/qformer_stage1`, `ckpt/qformer_stage2_vicuna`).

Nếu chưa có (4), chạy các bước R0–R2 trong `notebooks/reproduce_plain_qformer.ipynb` **một lần** rồi quay lại đây.


## 0. Kiểm tra tiên quyết
Dừng sớm nếu thiếu checkpoint tiền huấn luyện (tránh chạy nửa chừng rồi lỗi).


In [ ]:
import os
need = [
    "/content/CoQLLM/ckpt/qformer_stage1",
    "/content/CoQLLM/ckpt/qformer_stage2_vicuna",
]
# MF path (pretrained_path) — in nếu tồn tại thư mục mf
mf_dir = "/content/CoQLLM/ckpt/mf"
missing = [p for p in need if not os.path.exists(p)]
print("MF dir exists:", os.path.exists(mf_dir), "->", mf_dir)
for p in need:
    print("exists:", os.path.exists(p), "->", p)
assert not missing, f"THIEU checkpoint tien huan luyen: {missing}. Chay R0-R2 trong reproduce_plain_qformer.ipynb truoc."
print("\nOK — du dieu kien chay Stage-3 cho cac seed.")


---
## Seed 42
Chạy tuần tự: Bước 1 (LoRA) → Bước 2 (Q-Former + điều kiện hóa, arm `B3-qformer-ucq`) → eval test.
Kết quả AUC/uAUC in ở cuối ô eval (dòng `[test] ...`).


In [ ]:
# --- Seed 42 · Bước 1: LoRA (chỉ prompt văn bản) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step1_lora.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc run.seed=42


In [ ]:
# --- Seed 42 · Bước 2: Q-Former + điều kiện hóa người dùng (B3-qformer-ucq) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step2_cie.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc run.seed=42


In [ ]:
# --- Seed 42 · Eval trên test (đọc AUC/uAUC ở dòng [test] ...) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --step 2 \
  --options run.seed=42


---
## Seed 123
Chạy tuần tự: Bước 1 (LoRA) → Bước 2 (Q-Former + điều kiện hóa, arm `B3-qformer-ucq`) → eval test.
Kết quả AUC/uAUC in ở cuối ô eval (dòng `[test] ...`).


In [ ]:
# --- Seed 123 · Bước 1: LoRA (chỉ prompt văn bản) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step1_lora.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc run.seed=123


In [ ]:
# --- Seed 123 · Bước 2: Q-Former + điều kiện hóa người dùng (B3-qformer-ucq) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step2_cie.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc run.seed=123


In [ ]:
# --- Seed 123 · Eval trên test (đọc AUC/uAUC ở dòng [test] ...) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --step 2 \
  --options run.seed=123


---
## Seed 7
Chạy tuần tự: Bước 1 (LoRA) → Bước 2 (Q-Former + điều kiện hóa, arm `B3-qformer-ucq`) → eval test.
Kết quả AUC/uAUC in ở cuối ô eval (dòng `[test] ...`).


In [ ]:
# --- Seed 7 · Bước 1: LoRA (chỉ prompt văn bản) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step1_lora.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc run.seed=7


In [ ]:
# --- Seed 7 · Bước 2: Q-Former + điều kiện hóa người dùng (B3-qformer-ucq) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step2_cie.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc run.seed=7


In [ ]:
# --- Seed 7 · Eval trên test (đọc AUC/uAUC ở dòng [test] ...) ---
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --step 2 \
  --options run.seed=7


---
## Tổng hợp mean ± std
Điền AUC/uAUC của split **test** (lấy từ 3 ô eval ở trên) vào `results` rồi chạy ô dưới.


In [ ]:
import numpy as np

# Dien so tu 3 o eval (split "test"):
results = {
    42:  {"auc": None, "uauc": None},
    123: {"auc": None, "uauc": None},
    7:   {"auc": None, "uauc": None},
}

assert all(v["auc"] is not None and v["uauc"] is not None for v in results.values()), \
    "Hay dien du AUC/uAUC cho ca 3 seed truoc khi chay."

auc  = np.array([results[s]["auc"]  for s in results])
uauc = np.array([results[s]["uauc"] for s in results])

def fmt(x):
    # dau phay thap phan kieu luan van
    return (f"{x.mean():.4f} ± {x.std(ddof=1):.4f}").replace(".", ",")

print(f"Seeds: {list(results)}")
print(f"AUC  (test): {fmt(auc)}")
print(f"uAUC (test): {fmt(uauc)}")
print()
print("Cho bai bao: bao cao 'mean ± std (n=3 seeds: 42/123/7)'.")


### (Tùy chọn) Phương sai cho nhánh KHÔNG điều kiện hóa (B2)
Nếu muốn củng cố luôn ablation điều kiện hóa qua nhiều seed, thêm `model.qformer_config.user_conditioned=False`
và `run.qformer_stage3_step2.arm=B2-qformer` vào ô Bước 2 + ô eval, cho từng seed. Thư mục cũng tự tách theo seed.
